In [4]:
# --- Célula Extra: Transfer Learning ---
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Carregar ResNet50 pré-treinado (sem a camada de classificação final)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congelar as camadas do ResNet
for layer in base_model.layers:
  layer.trainable = False

# Adicionar camadas customizadas
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x) # 10 classes (Fashion MNIST adaptado)

# Criar o modelo final
model_transfer = Model(inputs=base_model.input, outputs=predictions)

model_transfer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Modelo com Transfer Learning pronto!")
print(f"Total de parâmetros: {model_transfer.count_params()}")
print(f"Parâmetros treináveis: {sum([tf.size(w).numpy() for w in model_transfer.trainable_weights])}")

Modelo com Transfer Learning pronto!
Total de parâmetros: 23851274
Parâmetros treináveis: 263562


### Resumo do Modelo de Transfer Learning

Nesta configuração, utilizamos a técnica de **Transfer Learning** com as seguintes características:

1.  **Base Model**: Utilizamos a **ResNet50** pré-treinada no dataset ImageNet. O parâmetro `include_top=False` remove a camada original de classificação, permitindo que usemos apenas os extratores de características.
2.  **Congelamento (Freezing)**: Definimos `layer.trainable = False` para todas as camadas da ResNet50. Isso garante que os pesos aprendidos no ImageNet não sejam alterados durante o treinamento inicial, economizando tempo computacional e evitando o esquecimento catastrófico.
3.  **Camadas Customizadas**:
    *   **GlobalAveragePooling2D**: Reduz a dimensionalidade espacial dos mapas de características.
    *   **Dense (128 units)**: Uma camada intermediária para aprender relações específicas do novo dataset.
    *   **Dense (10 units)**: A camada de saída com ativação *softmax*, configurada para classificar 10 categorias diferentes.

O modelo resultante possui mais de 23 milhões de parâmetros, mas apenas cerca de **263 mil** são treináveis, o que torna o treinamento muito mais rápido.